# MERRA-2 Data Verification and Quality Check

**Purpose**: Verify that MERRA-2 hourly data is correctly structured for the experimental design.

**Data Sources**:
- MERRA-2 daily files: `data/daily_data/`
- Cattle slaughter data: `data/cattle_data/cattle_data_clean.csv`
- Region masks: `data/masks/region_mask.nc`

**Key Checks**:
1. Temporal coverage (1984-2025)
2. Hourly resolution and time encoding
3. Spatial coverage (US Lower 48)
4. Variable availability (T2M, VPD)
5. Data quality (missing values, reasonable ranges)
6. File completeness (gaps in daily files)
7. Cattle data alignment
8. Region mask compatibility

**Dependencies**:
- Uses `config.py` for all path definitions
- Imports `load_cattle_data()` from `src.data_loading` module

This notebook ensures the raw MERRA-2 data is ready for predictor construction.

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import sys

# Add parent directory to path for config import
# Since we're in notebooks/03_analysis/, need to go up two levels to reach research/
sys.path.append('../..')
import config

# Import from src/ modules
from src.data_loading import load_cattle_data

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

print(f"MERRA-2 daily data directory: {config.DAILY_DATA_DIR}")
print(f"Cattle data file: {config.CATTLE_DATA_FILE}")
print(f"Region mask file: {config.MASK_FILE}")
print("\n✓ Imports from src/ modules loaded")
print("\n" + "="*80)
print("Directory Check:")
print("="*80)
print(f"Daily data directory exists: {config.DAILY_DATA_DIR.exists()}")
print(f"Cattle data file exists: {config.CATTLE_DATA_FILE.exists()}")
print(f"Region mask file exists: {config.MASK_FILE.exists()}")
if not config.DAILY_DATA_DIR.exists():
    print("\n⚠️  NOTE: Run MERRA-2 processing notebooks first to populate data/daily_data/")
if not config.CATTLE_DATA_FILE.exists():
    print("\n⚠️  NOTE: Ensure cattle data is in data/cattle_data/cattle_data_clean.csv")
if not config.MASK_FILE.exists():
    print("\n⚠️  NOTE: Ensure region mask is in data/masks/region_mask.nc")

## 1. Check File Coverage

Verify we have daily files for the full period and identify any gaps.

In [ ]:
# List all daily files
daily_files = sorted(config.DAILY_DATA_DIR.glob('merra2_us_*.nc'))

print(f"Total files found: {len(daily_files):,}")

if len(daily_files) > 0:
    # Extract dates from filenames
    file_dates = []
    for f in daily_files:
        date_str = f.stem.split('_')[-1]  # Extract YYYYMMDD
        file_dates.append(pd.to_datetime(date_str, format='%Y%m%d'))
    
    file_dates = sorted(file_dates)
    
    print(f"\nFirst file: {file_dates[0].strftime('%Y-%m-%d')}")
    print(f"Last file: {file_dates[-1].strftime('%Y-%m-%d')}")
    print(f"Total days: {len(file_dates):,}")
    
    # Check for gaps
    expected_dates = pd.date_range(file_dates[0], file_dates[-1], freq='D')
    missing_dates = set(expected_dates) - set(file_dates)
    
    if len(missing_dates) > 0:
        print(f"\n⚠️  WARNING: {len(missing_dates)} missing dates found:")
        missing_sorted = sorted(list(missing_dates))[:20]  # Show first 20
        for d in missing_sorted:
            print(f"  - {d.strftime('%Y-%m-%d')}")
        if len(missing_dates) > 20:
            print(f"  ... and {len(missing_dates) - 20} more")
    else:
        print("\n✓ No gaps found - complete daily coverage!")
else:
    print("\n⚠️  ERROR: No MERRA-2 files found!")
    print(f"    Expected location: {config.DAILY_DATA_DIR}")
    print(f"    Directory exists: {config.DAILY_DATA_DIR.exists()}")
    print("\n📋 To populate this directory:")
    print("    1. Run notebooks/01_data_acquisition/ notebooks to download MERRA-2 data")
    print("    2. Or run process_merra2_cdo.ipynb to process downloaded data")
    # Initialize empty list for later cells
    file_dates = []

## 2. Examine File Structure

Check a sample file to verify:
- 24 hourly timesteps per day
- T2M and VPD variables present
- Spatial dimensions (~51x95 for US Lower 48)
- Data types and compression

In [ ]:
# Pick a representative file from summer (high heat stress period)
sample_date = '20200615'  # June 15, 2020
sample_file = config.DAILY_DATA_DIR / f'merra2_us_{sample_date}.nc'

if sample_file.exists():
    ds = xr.open_dataset(sample_file)
    
    print("Dataset structure:")
    print(ds)
    
    print("\n" + "="*80)
    print("Data Validation Checks:")
    print("="*80)
    
    # Check dimensions
    time_steps = len(ds.time)
    lat_size = len(ds.lat)
    lon_size = len(ds.lon)
    
    print(f"\n✓ Time dimension: {time_steps} steps (expected: 24)")
    print(f"✓ Latitude dimension: {lat_size} points")
    print(f"✓ Longitude dimension: {lon_size} points")
    
    # Check variables
    has_t2m = 'T2M' in ds.variables
    has_vpd = 'VPD' in ds.variables
    
    print(f"\n✓ T2M variable present: {has_t2m}")
    print(f"✓ VPD variable present: {has_vpd}")
    
    # Check spatial extent
    print(f"\nSpatial coverage:")
    print(f"  Latitude: {ds.lat.values.min():.2f}° to {ds.lat.values.max():.2f}°")
    print(f"  Longitude: {ds.lon.values.min():.2f}° to {ds.lon.values.max():.2f}°")
    print(f"  Expected: Lat [24, 49], Lon [-125, -66] for US Lower 48")
    
    ds.close()
else:
    print(f"⚠️  Sample file not found: {sample_file}")
    print("   Please ensure MERRA-2 processing has been run.")

## 3. Check Data Quality

Verify temperature and VPD values are in reasonable ranges and check for missing data.

In [ ]:
if sample_file.exists():
    ds = xr.open_dataset(sample_file)
    
    # Extract variables
    t2m = ds['T2M']
    vpd = ds['VPD']
    
    print("Temperature (T2M) Statistics:")
    print(f"  Min: {float(t2m.min()):.2f}°C")
    print(f"  Max: {float(t2m.max()):.2f}°C")
    print(f"  Mean: {float(t2m.mean()):.2f}°C")
    print(f"  Std: {float(t2m.std()):.2f}°C")
    
    print("\nVapor Pressure Deficit (VPD) Statistics:")
    print(f"  Min: {float(vpd.min()):.3f} kPa")
    print(f"  Max: {float(vpd.max()):.3f} kPa")
    print(f"  Mean: {float(vpd.mean()):.3f} kPa")
    print(f"  Std: {float(vpd.std()):.3f} kPa")
    
    # Check for missing/invalid values
    t2m_missing = np.isnan(t2m.values).sum()
    vpd_missing = np.isnan(vpd.values).sum()
    
    print(f"\nMissing values:")
    print(f"  T2M: {t2m_missing:,} ({100*t2m_missing/t2m.size:.3f}%)")
    print(f"  VPD: {vpd_missing:,} ({100*vpd_missing/vpd.size:.3f}%)")
    
    # Check for unrealistic values
    t2m_too_cold = (t2m < -50).sum().values
    t2m_too_hot = (t2m > 60).sum().values
    vpd_negative = (vpd < 0).sum().values
    vpd_too_high = (vpd > 10).sum().values
    
    print(f"\nData quality flags:")
    print(f"  T2M < -50°C: {t2m_too_cold:,} points")
    print(f"  T2M > 60°C: {t2m_too_hot:,} points")
    print(f"  VPD < 0 kPa: {vpd_negative:,} points")
    print(f"  VPD > 10 kPa: {vpd_too_high:,} points")
    
    if t2m_too_cold + t2m_too_hot + vpd_negative + vpd_too_high == 0:
        print("\n✓ All values within reasonable ranges!")
    
    ds.close()

## 4. Visualize Hourly Patterns

Plot a diurnal cycle to verify hourly structure is correct.

In [ ]:
if sample_file.exists():
    ds = xr.open_dataset(sample_file)
    
    # Calculate spatial means for each hour
    t2m_hourly = ds['T2M'].mean(dim=['lat', 'lon']).values
    vpd_hourly = ds['VPD'].mean(dim=['lat', 'lon']).values
    hours = np.arange(24)
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Temperature
    ax1.plot(hours, t2m_hourly, marker='o', linewidth=2, markersize=6, color='#E74C3C')
    ax1.axvspan(20, 24, alpha=0.2, color='navy', label='Nighttime (8pm-midnight)')
    ax1.axvspan(0, 6, alpha=0.2, color='navy')
    ax1.axvspan(8, 20, alpha=0.2, color='gold', label='Daytime (8am-8pm)')
    ax1.set_xlabel('Hour of Day (UTC)', fontsize=12)
    ax1.set_ylabel('Temperature (°C)', fontsize=12)
    ax1.set_title(f'Diurnal Temperature Cycle - {sample_date}', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.set_xticks(hours)
    
    # VPD
    ax2.plot(hours, vpd_hourly, marker='s', linewidth=2, markersize=6, color='#3498DB')
    ax2.axvspan(12, 18, alpha=0.2, color='orange', label='VPD window (12pm-6pm)')
    ax2.set_xlabel('Hour of Day (UTC)', fontsize=12)
    ax2.set_ylabel('VPD (kPa)', fontsize=12)
    ax2.set_title(f'Diurnal VPD Cycle - {sample_date}', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    ax2.set_xticks(hours)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Diurnal patterns show expected behavior:")
    print(f"  - Temperature peak at hour {np.argmax(t2m_hourly)} (~{t2m_hourly.max():.1f}°C)")
    print(f"  - Temperature minimum at hour {np.argmin(t2m_hourly)} (~{t2m_hourly.min():.1f}°C)")
    print(f"  - VPD peak at hour {np.argmax(vpd_hourly)} (~{vpd_hourly.max():.2f} kPa)")
    
    ds.close()

## 5. Check Cattle Data Coverage

Verify cattle slaughter data aligns with MERRA-2 temporal coverage.

In [ ]:
# Load cattle data using src function
cattle_df = load_cattle_data(config.CATTLE_DATA_FILE)

if cattle_df is not None:
    print("Cattle Data Summary:")
    print(f"  Total weeks: {len(cattle_df):,}")
    print(f"  Date range: {cattle_df['date'].min()} to {cattle_df['date'].max()}")
    print(f"  Regions: {[col for col in cattle_df.columns if 'region' in col]}")
    
    # Check overlap with MERRA-2 data
    if len(daily_files) > 0 and len(file_dates) > 0:
        merra_start = file_dates[0]
        merra_end = file_dates[-1]
        cattle_start = cattle_df['date'].min()
        cattle_end = cattle_df['date'].max()
        
        overlap_start = max(merra_start, cattle_start)
        overlap_end = min(merra_end, cattle_end)
        
        print(f"\nOverlap period:")
        print(f"  Start: {overlap_start}")
        print(f"  End: {overlap_end}")
        print(f"  Duration: {(overlap_end - overlap_start).days} days")
        
        # Calculate number of weeks in overlap
        n_weeks = len(cattle_df[(cattle_df['date'] >= overlap_start) & 
                                 (cattle_df['date'] <= overlap_end)])
        print(f"  Weeks available for analysis: {n_weeks:,}")
        
        if n_weeks > 1000:
            print("\n✓ Excellent temporal coverage for analysis!")
    else:
        print("\n⚠️  Cannot check overlap - MERRA-2 data not found")
    
    # Show sample of data
    print("\nSample of cattle data (first 5 rows):")
    display(cattle_df.head())
    
    # Check for missing values
    missing_counts = cattle_df.isnull().sum()
    if missing_counts.sum() > 0:
        print("\n⚠️  Missing values found:")
        print(missing_counts[missing_counts > 0])
    else:
        print("\n✓ No missing values in cattle data!")
else:
    print(f"⚠️  Cattle data file not found: {config.CATTLE_DATA_FILE}")
    print(f"\n📋 To populate this file:")
    print(f"    1. Place cattle_data_clean.csv in {config.CATTLE_DATA_DIR}/")
    print(f"    2. Ensure the file has columns: date, region_1, region_2, ..., region_10")

## 6. Regional Mapping Check

Verify we can properly map MERRA-2 grid cells to cattle regions using the region mask.

In [ ]:
if config.MASK_FILE.exists():
    mask_ds = xr.open_dataset(config.MASK_FILE)
    
    print("Region Mask Information:")
    print(mask_ds)
    
    print("\nStates in focus regions (4 & 6):")
    for region_name, states in config.CATTLE_REGIONS.items():
        print(f"\n{region_name.upper()}:")
        for state in states:
            state_id = config.FOCUS_STATES.get(state)
            if state_id:
                print(f"  - {state} (ID: {state_id}, Abbr: {config.STATE_ABBRS[state_id]})")
    
    # Check mask dimensions match MERRA-2
    if sample_file.exists():
        ds = xr.open_dataset(sample_file)
        
        print(f"\nDimension matching:")
        print(f"  MERRA-2 grid: {len(ds.lat)} lat × {len(ds.lon)} lon")
        print(f"  Region mask: {len(mask_ds.lat)} lat × {len(mask_ds.lon)} lon")
        
        if len(ds.lat) == len(mask_ds.lat) and len(ds.lon) == len(mask_ds.lon):
            print("\n✓ Dimensions match - ready for regional aggregation!")
        else:
            print("\n⚠️  WARNING: Dimension mismatch - check processing")
        
        ds.close()
    else:
        print(f"\n⚠️  Cannot verify dimensions - sample MERRA-2 file not found")
    
    mask_ds.close()
else:
    print(f"⚠️  Region mask file not found: {config.MASK_FILE}")
    print(f"\n📋 To create the region mask:")
    print(f"    1. Run create_region_mask_pointwise.py script")
    print(f"    2. Mask will be created at {config.MASK_FILE}")
    print(f"    3. Mask defines state/region assignments for {len(config.STATE_NAMES)} states")

## 7. Summary

Final verification checklist for experimental design readiness.

In [ ]:
print("="*80)
print("DATA VERIFICATION CHECKLIST")
print("="*80)

if len(daily_files) > 0:
    print("\n✓ MERRA-2 Data:")
    print(f"  - Daily files: {len(daily_files):,} files")
    print(f"  - Coverage: {file_dates[0].year}-{file_dates[-1].year} ({len(file_dates):,} days)")
    print(f"  - Variables: T2M (°C), VPD (kPa)")
    print(f"  - Temporal resolution: 24 hourly timesteps per day")
    if 'lat_size' in locals() and 'lon_size' in locals():
        print(f"  - Spatial resolution: ~0.5° × 0.625° ({lat_size} × {lon_size})")
else:
    print("\n✗ MERRA-2 Data: Not found")
    print(f"  - Expected location: {config.DAILY_DATA_DIR}")
    print(f"  - Please run data processing notebooks to populate data/daily_data/")

if config.CATTLE_DATA_FILE.exists() and 'cattle_df' in locals() and cattle_df is not None:
    print("\n✓ Cattle Slaughter Data:")
    print(f"  - Weekly data: {len(cattle_df):,} weeks")
    print(f"  - Coverage: {cattle_df['date'].min().year}-{cattle_df['date'].max().year}")
    print(f"  - Regions: 10 USDA regions")
    print(f"  - Loaded using: src.data_loading.load_cattle_data()")
else:
    print("\n✗ Cattle Slaughter Data: Not found")
    print(f"  - Expected location: {config.CATTLE_DATA_FILE}")

if config.MASK_FILE.exists():
    print("\n✓ Region Mask:")
    print(f"  - States defined: {len(config.STATE_NAMES)}")
    print(f"  - Focus states: {len(config.FOCUS_STATES)}")
    print(f"  - Cattle regions: {list(config.CATTLE_REGIONS.keys())}")
else:
    print("\n✗ Region Mask: Not found")
    print(f"  - Expected location: {config.MASK_FILE}")

print("\n" + "="*80)
if len(daily_files) > 0 and config.CATTLE_DATA_FILE.exists() and config.MASK_FILE.exists():
    print("READY FOR EXPERIMENTAL DESIGN SETUP")
else:
    print("SETUP INCOMPLETE - REVIEW CHECKLIST ABOVE")
print("="*80)
print("\nNext steps:")
print("1. Ensure all data directories are populated (see checklist above)")
print("2. Run '01b_experimental_design_from_processed.ipynb' to construct predictors")
print("3. TAS (Thermal Acclimation State) calculated using src.thermal_acclimation")
print("4. Aggregate to weekly and merge with cattle data using src.weekly_operations")